# Emotion Classification with DistilBERT

Fine-tunes a pretrained transformer to classify short English texts into six emotions, and benchmarks it against a classical TF-IDF + Logistic Regression baseline.

**Methodology**
- **Baseline first:** a simple, fast model sets the bar the transformer must beat to justify its extra cost.
- **Macro-F1 as the headline metric:** the classes are imbalanced, so accuracy alone would hide weak performance on rare emotions.
- **Clean evaluation:** hyperparameters are chosen on the validation set; the test set is used exactly once, at the end.
- **Error analysis:** the most confident mistakes are inspected by hand to separate genuine model errors from ambiguous or noisy labels.

**How to run:** run the cells top to bottom. Start with `SMOKE_TEST = True` (a fast sanity check), then set it to `False` for the real run. Full training is slow on a laptop CPU, so use a GPU runtime such as Google Colab (T4).

# 1. Setup and configuration

In [ ]:
import sys, subprocess

# Colab starts from a clean environment, so install what is missing. Does nothing locally.
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "datasets", "accelerate", "scikit-learn", "gradio"], check=True)

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
from IPython.display import display
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

sns.set_theme(style="whitegrid")

# ---------------------------- Configuration ----------------------------
# True  -> 500 training samples, 1 epoch: verifies the pipeline end to end in minutes.
# False -> full training run (use a GPU).
SMOKE_TEST = True

HUB_DATASET = "dair-ai/emotion"          # ships with fixed train / validation / test splits
MODEL_NAME = "distilbert-base-uncased"   # ~40% smaller and ~60% faster than BERT-base, with most of its accuracy
MAX_LENGTH = 64                          # tweets are short, so 64 tokens covers nearly every example
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5                     # standard range for fine-tuning transformers: 1e-5 to 5e-5
SEED = 42                                # fixed seed so results are reproducible

HUB_MODEL_ID = "your-username/distilbert-emotion"   # your Hugging Face username (used for deployment)

DEMO_EXAMPLES = [
    "I can't believe I finally got the job, I'm over the moon!",
    "I miss my grandmother so much it hurts.",
    "How dare you lie to me again, I am furious.",
    "I heard a noise downstairs and my heart is pounding.",
]
# Deliberately hard inputs used to probe the model's limits (sarcasm, negation).
TRICKY_EXAMPLES = [
    "Oh great, another Monday. I'm just thrilled.",
    "I am not happy about this at all.",
    "Wow, I never expected this to happen!",
    "I love spending quiet evenings with you.",
]
# -----------------------------------------------------------------------

# A smoke test saves to its own folder so it can never overwrite the real model.
MODEL_DIR = "models/distilbert-emotion" + ("-smoke-test" if SMOKE_TEST else "")
for folder in ("results", "models", "app"):
    os.makedirs(folder, exist_ok=True)

set_seed(SEED)
print(f"Smoke test: {SMOKE_TEST} | GPU available: {torch.cuda.is_available()} | model dir: {MODEL_DIR}")

# 2. Load the data

The dataset ships with three fixed splits, each with a clear role:
- **train:** the model learns from it
- **validation:** used for model selection and tuning
- **test:** held out until the final evaluation, so the reported score is an honest estimate of real-world performance

In [ ]:
ds = load_dataset(HUB_DATASET)

LABEL_NAMES = ds["train"].features["label"].names
NUM_LABELS = len(LABEL_NAMES)
id2label = dict(enumerate(LABEL_NAMES))
label2id = {name: i for i, name in id2label.items()}

# pandas copies are used for exploration and the baseline; the Hugging Face datasets are used for the transformer.
train_df, val_df, test_df = (ds[s].to_pandas() for s in ("train", "validation", "test"))
for d in (train_df, val_df, test_df):
    d["emotion"] = d["label"].map(id2label)
    d["n_words"] = d["text"].str.split().str.len()

print("Classes:", LABEL_NAMES)
print(f"Rows -> train: {len(train_df)} | validation: {len(val_df)} | test: {len(test_df)}")
train_df.head()

# 3. Explore the data

Understanding class balance, text length and data quality up front guides the metric choice and `MAX_LENGTH`, and surfaces problems (such as leakage) before they inflate results.

In [ ]:
counts = train_df["emotion"].value_counts()
p95 = train_df["n_words"].quantile(0.95)
print(counts, f"\n\n95% of texts have {p95:.0f} words or fewer")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], color="#4C72B0")
axes[0].set_title("Training examples per class")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")

axes[1].hist(train_df["n_words"], bins=30, color="#4C72B0")
axes[1].axvline(p95, color="#C44E52", linestyle="--", label=f"95th percentile = {p95:.0f} words")
axes[1].set_title("Words per text")
axes[1].set_xlabel("Number of words")
axes[1].legend()

plt.tight_layout()
plt.savefig("results/eda.png", dpi=150)
plt.show()

In [ ]:
# Data-quality checks. Overlap between splits (leakage) would inflate test scores;
# one text carrying several labels indicates label noise.
print("Missing values in train:", int(train_df.isna().sum().sum()))
print("Duplicate texts in train:", train_df["text"].duplicated().sum())
print("Texts shared by train and test:", len(set(train_df["text"]) & set(test_df["text"])))
print("Texts shared by train and validation:", len(set(train_df["text"]) & set(val_df["text"])))
print("Texts with conflicting labels:", (train_df.groupby("text")["label"].nunique() > 1).sum())

# Reading raw examples by hand is the quickest way to judge how clean the labels are.
for name in LABEL_NAMES:
    print(f"\n===== {name.upper()} =====")
    for text in train_df[train_df["emotion"] == name].sample(3, random_state=SEED)["text"]:
        print("  -", text)

**Observations** *(write your own after reading the output above: which classes dominate, which examples look ambiguous or mislabeled?)*

# 4. Baseline: TF-IDF + Logistic Regression

A baseline answers the question "does the transformer actually help?". TF-IDF turns text into weighted word and word-pair counts, and Logistic Regression learns a linear classifier on top. The regularisation strength `C` is tuned on the **validation** set only.

**Metrics used**
- **Precision / recall:** how often a predicted class is right, and how many true cases of that class are found.
- **F1:** the harmonic mean of the two. **Macro-F1** averages F1 over classes with equal weight, so rare classes count as much as common ones.

In [ ]:
# The vocabulary is learned from the training set only, so no information leaks from validation or test.
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
y_train, y_val = train_df["label"], val_df["label"]

candidates = {C: LogisticRegression(C=C, max_iter=2000).fit(X_train, y_train) for C in [0.1, 1, 10, 100]}
val_f1 = {C: f1_score(y_val, m.predict(X_val), average="macro") for C, m in candidates.items()}
for C, score in val_f1.items():
    print(f"C = {C:<5} validation macro-F1 = {score:.4f}")

best_C = max(val_f1, key=val_f1.get)
baseline = candidates[best_C]
print(f"\nSelected C = {best_C}")
print(classification_report(y_val, baseline.predict(X_val), target_names=LABEL_NAMES, digits=3))

joblib.dump((vectorizer, baseline), "models/baseline_tfidf_logreg.joblib")

# 5. Fine-tune DistilBERT

DistilBERT is already pretrained on a large English corpus. Fine-tuning adapts it to our task by training a new classification head, and updating the pretrained weights, on our labeled examples. This needs far less data and compute than training from scratch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    # Truncate to MAX_LENGTH; padding is applied per batch later (dynamic padding is faster than padding everything up front).
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds, val_ds = ds["train"], ds["validation"]
if SMOKE_TEST:
    train_ds = train_ds.shuffle(seed=SEED).select(range(500))
    val_ds = val_ds.select(range(200))
    epochs = 1
else:
    epochs = EPOCHS

train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)

# Example of what the model actually sees: subword tokens mapped to integer ids.
sample = train_df["text"].iloc[0]
print("Text  :", sample)
print("Tokens:", tokenizer.tokenize(sample))

In [ ]:
# The warning about "newly initialized weights" is expected: it is the new classification head we are about to train.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

training_args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=epochs,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,                  # mild regularisation against overfitting
    eval_strategy="epoch",              # validate after every epoch...
    save_strategy="epoch",
    load_best_model_at_end=True,        # ...and keep the best epoch by macro-F1, not simply the last one
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
print("Validation metrics:", trainer.evaluate())

# Learning curves: a rising validation loss while training loss keeps falling signals overfitting.
log = trainer.state.log_history
train_pts = [(l["step"], l["loss"]) for l in log if "loss" in l]
eval_pts = [(l["epoch"], l["eval_loss"], l["eval_macro_f1"]) for l in log if "eval_loss" in l]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(*zip(*train_pts))
axes[0].set(title="Training loss", xlabel="step")
axes[1].plot([p[0] for p in eval_pts], [p[1] for p in eval_pts], marker="o", label="validation loss")
axes[1].plot([p[0] for p in eval_pts], [p[2] for p in eval_pts], marker="s", label="validation macro-F1")
axes[1].set(title="Validation scores per epoch", xlabel="epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150)
plt.show()

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
with open(os.path.join(MODEL_DIR, "training_log.json"), "w") as f:
    json.dump(log, f, indent=2, default=float)

# 6. Final evaluation on the test set

This is the only time the test set is used. Both models are scored on it and nothing is tuned afterwards.

In [ ]:
def predict_proba(texts, batch_size=64):
    """Return class probabilities as an array of shape (n_texts, n_classes)."""
    model.eval()
    chunks = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(list(texts[i:i + batch_size]), truncation=True, max_length=MAX_LENGTH,
                            padding=True, return_tensors="pt").to(model.device)
            chunks.append(torch.softmax(model(**enc).logits, dim=-1).cpu().numpy())
    return np.vstack(chunks)

y_true = test_df["label"].values
base_pred = baseline.predict(vectorizer.transform(test_df["text"]))
bert_probs = predict_proba(test_df["text"].tolist())
bert_pred = bert_probs.argmax(axis=1)

results = pd.DataFrame({
    name: {"accuracy": accuracy_score(y_true, pred),
           "macro_f1": f1_score(y_true, pred, average="macro"),
           "weighted_f1": f1_score(y_true, pred, average="weighted")}
    for name, pred in [("TF-IDF + Logistic Regression", base_pred), ("DistilBERT (fine-tuned)", bert_pred)]
}).T.round(4)
results.to_csv("results/test_results.csv")
display(results)

if SMOKE_TEST:
    print("SMOKE TEST: these numbers come from a tiny run. Do not report them; set SMOKE_TEST = False and rerun.")

In [ ]:
# Per-class F1 shows *where* the gains come from, which the overall averages hide.
print("=== Baseline ===")
print(classification_report(y_true, base_pred, target_names=LABEL_NAMES, digits=3, zero_division=0))
print("=== DistilBERT ===")
print(classification_report(y_true, bert_pred, target_names=LABEL_NAMES, digits=3, zero_division=0))

labels = range(NUM_LABELS)
per_class = pd.DataFrame({
    "Baseline": f1_score(y_true, base_pred, average=None, labels=labels, zero_division=0),
    "DistilBERT": f1_score(y_true, bert_pred, average=None, labels=labels, zero_division=0)},
    index=LABEL_NAMES)
ax = per_class.plot(kind="bar", figsize=(8, 4), color=["#8C8C8C", "#4C72B0"])
ax.set(title="F1 per class (test set)", ylabel="F1", ylim=(0, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("results/per_class_f1.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrices: rows are true classes, columns are predictions; off-diagonal cells are the mix-ups.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (title, pred) in zip(axes, [("Baseline", base_pred), ("DistilBERT", bert_pred)]):
    ConfusionMatrixDisplay.from_predictions(y_true, pred, display_labels=LABEL_NAMES,
                                            xticks_rotation=45, cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"{title}: test confusion matrix")
plt.tight_layout()
plt.savefig("results/confusion_matrices_test.png", dpi=150)
plt.show()

### Error analysis

Aggregate metrics say how often the model is wrong; error analysis says **why**. Examples the model gets wrong *with high confidence* are the most informative: they point either to a systematic blind spot or to a questionable label.

In [ ]:
bert_wrong = bert_pred != y_true
base_wrong = base_pred != y_true
print(f"DistilBERT errors: {bert_wrong.sum()} of {len(test_df)}")
print(f"  also wrong in baseline: {(bert_wrong & base_wrong).sum()} | only DistilBERT wrong: {(bert_wrong & ~base_wrong).sum()}")
print(f"Errors fixed relative to baseline: {(~bert_wrong & base_wrong).sum()}")

test_df["bert_pred"] = [id2label[i] for i in bert_pred]
test_df["bert_conf"] = bert_probs.max(axis=1).round(3)
errors = test_df[bert_wrong]

print("\nMost frequent confusions (true -> predicted):")
print(errors.groupby(["emotion", "bert_pred"]).size().sort_values(ascending=False).head(8))

# Export the 50 most confident mistakes for manual review.
review = (errors.sort_values("bert_conf", ascending=False)
                .head(50)[["text", "emotion", "bert_pred", "bert_conf"]]
                .assign(category="")
                .reset_index(drop=True))
review.to_csv("results/error_analysis_to_fill.csv", index=False)
display(review.head(15))

**Your turn:** open `results/error_analysis_to_fill.csv` and label each row in the `category` column, for example:
`ambiguous` (a human could argue either way), `label looks wrong`, `needs context / sarcasm`, or `model mistake`.
Count the categories and summarise what you found in 4-5 sentences. This becomes the "Error analysis" section of your README.

**My findings:** *(write here)*

In [ ]:
# Probe the model with hand-written sentences, including sarcasm and negation, which bag-of-words style cues handle poorly.
sentences = DEMO_EXAMPLES + TRICKY_EXAMPLES
for text, p in zip(sentences, predict_proba(sentences)):
    print(f"{id2label[int(p.argmax())]:<10} ({p.max():.0%})  <-  {text}")

# 7. Interactive demo

In [ ]:
import gradio as gr

def predict_emotions(text):
    if not text or not text.strip():
        return {}
    probs = predict_proba([text])[0]
    return {id2label[i]: float(probs[i]) for i in range(NUM_LABELS)}

demo = gr.Interface(
    fn=predict_emotions,
    inputs=gr.Textbox(lines=3, label="Type a sentence"),
    outputs=gr.Label(num_top_classes=NUM_LABELS, label="Prediction"),
    title="Emotion Classifier",
    examples=[[e] for e in DEMO_EXAMPLES],
    flagging_mode="never",
)
demo.launch()      # call demo.close() when you are finished

### Deployment files for Hugging Face Spaces

The next cell writes `app/app.py` and `app/requirements.txt`. Upload them to a new Space (SDK: Gradio, free CPU). The app loads the model from the Hub, so push the model first (section 8.1).

In [ ]:
APP = '''
import os
import gradio as gr
from transformers import pipeline

# Use the local model when it exists (development); otherwise load from the Hugging Face Hub (Spaces).
LOCAL_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "models", "distilbert-emotion")
MODEL_ID = LOCAL_DIR if os.path.isdir(LOCAL_DIR) else "__HUB_ID__"

classifier = pipeline("text-classification", model=MODEL_ID, top_k=None)
n_classes = len(classifier.model.config.id2label)


def predict(text):
    if not text or not text.strip():
        return {}
    scores = classifier(text, truncation=True, max_length=__MAX_LENGTH__)
    if scores and isinstance(scores[0], list):   # some library versions nest the output
        scores = scores[0]
    return {s["label"]: float(s["score"]) for s in scores}


demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, label="Type a sentence"),
    outputs=gr.Label(num_top_classes=n_classes),
    title="Emotion Classifier",
    examples=__EXAMPLES__,
    flagging_mode="never",
)

if __name__ == "__main__":
    demo.launch()
'''.strip() + "\n"

app_code = (APP.replace("__HUB_ID__", HUB_MODEL_ID)
               .replace("__MAX_LENGTH__", str(MAX_LENGTH))
               .replace("__EXAMPLES__", repr([[e] for e in DEMO_EXAMPLES])))

with open("app/app.py", "w") as f:
    f.write(app_code)
with open("app/requirements.txt", "w") as f:
    f.write("transformers\ntorch\n")
print("Wrote app/app.py and app/requirements.txt")

# 8. Publish and package

### 8.1 Upload the model to Hugging Face (after the real run only)
Create a **Write** token at huggingface.co (Settings, Access Tokens), uncomment the lines below and run.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub(HUB_MODEL_ID)
# tokenizer.push_to_hub(HUB_MODEL_ID)

### 8.2 Generate the README with your real numbers

In [ ]:
labels_str = ", ".join(LABEL_NAMES)
base_row = results.loc["TF-IDF + Logistic Regression"]
bert_row = results.loc["DistilBERT (fine-tuned)"]

README = f'''# Emotion Classification with DistilBERT

Fine-tuned DistilBERT to classify short texts into {NUM_LABELS} emotions ({labels_str}), benchmarked against a TF-IDF + Logistic Regression baseline.

**Live demo:** [add your Hugging Face Space link]
**Model:** [add your Hugging Face model link]

## Problem
[1-2 sentences: what is being classified, and who would use it?]

## Data
`{HUB_DATASET}`: {len(train_df)} train / {len(val_df)} validation / {len(test_df)} test. The classes are imbalanced (see `results/eda.png`), so I report **macro-F1**, which weights every class equally, alongside accuracy.

## Approach
1. **Baseline:** TF-IDF (unigrams and bigrams) + Logistic Regression, with `C = {best_C}` selected on the validation set.
2. **Main model:** `{MODEL_NAME}` fine-tuned for {epochs} epoch(s) (learning rate {LEARNING_RATE}, batch size {BATCH_SIZE}, max length {MAX_LENGTH}); best epoch chosen by validation macro-F1.
3. The test set was used once, for the final comparison.

## Results (test set)
| Model | Accuracy | Macro-F1 |
|---|---|---|
| TF-IDF + Logistic Regression | {base_row["accuracy"]:.3f} | {base_row["macro_f1"]:.3f} |
| DistilBERT (fine-tuned) | {bert_row["accuracy"]:.3f} | {bert_row["macro_f1"]:.3f} |

![Confusion matrices](results/confusion_matrices_test.png)
![Per-class F1](results/per_class_f1.png)

## Error analysis
[4-5 sentences from your hand-labeled errors: which classes are confused, how many errors were ambiguous, mislabeled, sarcastic or genuine model mistakes, and what you would try next.]

## How to run
```bash
pip install -r requirements.txt
jupyter notebook emotion_classifier.ipynb   # use Google Colab (GPU) for the training step
python app/app.py                           # launch the demo locally
```

## Limitations and next steps
- [e.g. English only, short texts; labels may be noisy]
- [e.g. try a larger model, class weights for rare classes, more data]

## Tech stack
Python, PyTorch, Hugging Face Transformers and Datasets, scikit-learn, pandas, Gradio
'''

with open("README_generated.md", "w") as f:
    f.write(README)
with open("requirements.txt", "w") as f:
    f.write("transformers\ndatasets\naccelerate\ntorch\nscikit-learn\npandas\nmatplotlib\nseaborn\ngradio\njoblib\n")
if not os.path.exists(".gitignore"):
    with open(".gitignore", "w") as f:
        f.write("venv/\n.env\n__pycache__/\n.ipynb_checkpoints/\ncheckpoints/\nmodels/\n*.safetensors\n*.bin\n*.zip\n.DS_Store\n")

print("Wrote README_generated.md (rename to README.md and fill in the [brackets]), requirements.txt and .gitignore.")
if SMOKE_TEST:
    print("SMOKE TEST: the numbers above are NOT real. Rerun with SMOKE_TEST = False before using them.")
print("\nResume bullet (use only numbers from the real run):")
print(f"  Fine-tuned DistilBERT for {NUM_LABELS}-class emotion classification, improving macro-F1 from "
      f"{base_row['macro_f1']:.2f} (TF-IDF baseline) to {bert_row['macro_f1']:.2f}; "
      f"deployed an interactive demo on Hugging Face Spaces.")

### 8.3 Colab only: download the outputs

In [ ]:
if ON_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("results", "zip", "results")
    shutil.make_archive("model", "zip", MODEL_DIR)
    for p in ("results.zip", "model.zip", "app/app.py", "app/requirements.txt",
              "README_generated.md", "requirements.txt"):
        files.download(p)

## Next steps
1. Fill in your error-analysis findings and the README `[brackets]`, then push the project to GitHub.
2. Deploy the demo on Hugging Face Spaces and add the link to your README and resume.